In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_tavily import TavilySearch
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
import gradio as gr
from IPython.display import display, Image


In [ ]:
load_dotenv(override=True)
google_api_key=os.getenv("GOOGLE_API_KEY")

In [ ]:
#llm=ChatOpenAI(model="llama3.2:1b",api_key="key", base_url="http://localhost:11434/v1")
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",api_key=google_api_key)
#memory=MemorySaver()

DB_path="memory.db"
conn=sqlite3.connect(DB_path, check_same_thread=False)
sql_memory=SqliteSaver(conn)


In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
graph_builder=StateGraph(State)

In [ ]:
tavily_search=TavilySearch(max_results=2)

@tool
def web_search(query: str):
    '''Use only for competator product information. Search the web to get  information about competator product like latest updates, market comparisions'''
    return tavily_search.invoke(query)

In [ ]:
DB_NAME="vector_db"
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embedding_model)
retriever=vectorstore.as_retriever()

@tool
def internal_product_rag(query: str):
    '''Use only to get informantion about internal product. Like product specifications and architecture'''
    docs=retriever.invoke(query)
    
    if not docs:
        return "no documents found"
    
    context= "\n\n".join([doc.page_content for doc in docs])

    return context

In [ ]:
tools=[web_search,internal_product_rag]
llm_with_tools=llm.bind_tools(tools)

In [ ]:
def chatbot(state: State):
    response=llm_with_tools.invoke(state["messages"])
    return { "messages" : [response]}

In [ ]:
graph_builder.add_node("chat_bot",chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "chat_bot")
graph_builder.add_conditional_edges("chat_bot",tools_condition)
graph_builder.add_edge("tools","chat_bot")


In [ ]:
graph=graph_builder.compile(checkpointer=sql_memory)
config = {"configurable": {"thread_id": "1"}}

In [ ]:
def chat_llm(user_msg: str, history):
    messages=[HumanMessage(content=user_msg)]
    state={"messages": messages}
    response=graph.invoke(state,config=config)
    return response["messages"][-1].content

In [ ]:
gr.ChatInterface(fn=chat_llm).launch()

In [ ]:
graph.get_state(config)